# 02 - Prepare Splits And Features

Notebook này chuẩn bị dữ liệu cho pipeline CK Isaac GR00T N1/N1.6.

Mục tiêu:

- Đọc 3 subset GR-1 đã tải từ `GR00T-X-Embodiment-Sim`.
- Kiểm tra `meta/`, `data/`, `videos/`.
- Đọc parquet trong `data/`.
- Detect episode/frame/action/state columns.
- Split train/test theo episode để tránh leakage.
- Lưu cache `.npy` và `.parquet` cho notebook train DiT/action head.

Notebook này chạy được trên CPU. GPU chỉ cần ở các notebook train/eval sau.


In [1]:
!pip install -q pandas pyarrow numpy tqdm


In [2]:
from pathlib import Path
import json
import random
import shutil
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
TRAIN_RATIO = 0.8

# Pipeline chính dùng toàn bộ frame. Nếu cần smoke test nhanh, tạm đổi thành 200.
MAX_FRAMES_PER_EPISODE = None

SUBSETS = [
    "gr1_arms_only.CanSort",
    "gr1_arms_waist.CanToDrawer",
    "gr1_arms_waist.CupToDrawer",
]

# Minh chứng tải dữ liệu từ các Kaggle notebook output hiện tại.
DOWNLOAD_EVIDENCE = [
    {
        "notebook_id": "notebook1700561abd",
        "subset": "gr1_arms_only.CanSort",
        "content": "data + meta + videos",
        "duration": "6 phút",
        "report": "download_report_1_5.json",
    },
    {
        "notebook_id": "notebookbcd29698a0",
        "subset": "gr1_arms_waist.CupToDrawer",
        "content": "data + meta + videos",
        "duration": "1 tiếng 40 phút",
        "report": "download_report_1_5.json",
    },
    {
        "notebook_id": "notebookb6c37dbbd0",
        "subset": "gr1_arms_waist.CanToDrawer",
        "content": "data + meta",
        "duration": "24 phút",
        "report": "download_report_6_10.json",
    },
    {
        "notebook_id": "notebookd3f6a10db4",
        "subset": "gr1_arms_waist.CanToDrawer",
        "content": "video supplement: observation.images.ego_view, chunks 0,1,2",
        "duration": "20 phút",
        "report": "video_supplement_report_CanToDrawer.json",
    },
]

OUTPUT_ROOT = Path("/kaggle/working/gr00t_prepared_3subsets")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print("Subsets:")
for subset in SUBSETS:
    print("-", subset)
print("\nDownload evidence:")
for item in DOWNLOAD_EVIDENCE:
    print(f"- {item['notebook_id']}: {item['subset']} | {item['content']} | {item['duration']}")
print("Output:", OUTPUT_ROOT)


Subsets:
- gr1_arms_only.CanSort
- gr1_arms_waist.CanToDrawer
- gr1_arms_waist.CupToDrawer

Download evidence:
- notebook1700561abd: gr1_arms_only.CanSort | data + meta + videos | 6 phút
- notebookbcd29698a0: gr1_arms_waist.CupToDrawer | data + meta + videos | 1 tiếng 40 phút
- notebookb6c37dbbd0: gr1_arms_waist.CanToDrawer | data + meta | 24 phút
- notebookd3f6a10db4: gr1_arms_waist.CanToDrawer | video supplement: observation.images.ego_view, chunks 0,1,2 | 20 phút
Output: /kaggle/working/gr00t_prepared_3subsets


In [3]:
def list_candidate_roots() -> List[Path]:
    """Find every gr00t_x_embodiment_sim* folder from Kaggle input and working dirs."""
    candidates = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        candidates.extend(base.rglob("gr00t_x_embodiment_sim*"))

    roots = []
    seen = set()
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        if not any((candidate / subset).exists() for subset in SUBSETS):
            continue
        key = str(candidate.resolve())
        if key not in seen:
            roots.append(candidate)
            seen.add(key)

    if not roots:
        raise FileNotFoundError(
            "Không tìm thấy gr00t_x_embodiment_sim*. Hãy Add Input cả 4 output datasets: "
            "CanSort, CupToDrawer, CanToDrawer data/meta, CanToDrawer video supplement."
        )
    return sorted(roots, key=lambda p: ("/kaggle/input" not in str(p), str(p)))


def collect_subset_sources(roots: List[Path]) -> Dict[str, Dict]:
    sources = {}
    for subset in SUBSETS:
        sources[subset] = {
            "data_meta_path": None,
            "video_paths": [],
            "all_paths": [],
        }

    for root in roots:
        for subset in SUBSETS:
            subset_path = root / subset
            if not subset_path.exists():
                continue
            sources[subset]["all_paths"].append(subset_path)

            data_dir = subset_path / "data"
            meta_dir = subset_path / "meta"
            videos_dir = subset_path / "videos"

            has_parquet = data_dir.exists() and any(data_dir.rglob("*.parquet"))
            if has_parquet and sources[subset]["data_meta_path"] is None:
                sources[subset]["data_meta_path"] = subset_path

            has_video = videos_dir.exists() and any(videos_dir.rglob("*.mp4"))
            if has_video:
                sources[subset]["video_paths"].append(subset_path)

    missing_data = [s for s, info in sources.items() if info["data_meta_path"] is None]
    if missing_data:
        raise FileNotFoundError(
            "Các subset thiếu data/*.parquet nên chưa prepare được: " + ", ".join(missing_data)
        )
    return sources


RAW_ROOTS = list_candidate_roots()
SUBSET_SOURCES = collect_subset_sources(RAW_ROOTS)

print("RAW_ROOTS:")
for root in RAW_ROOTS:
    print("-", root)

print("\nSUBSET_SOURCES:")
for subset, info in SUBSET_SOURCES.items():
    print("-", subset)
    print("  data/meta:", info["data_meta_path"])
    print("  video paths:", [str(p) for p in info["video_paths"]])


RAW_ROOTS:
- /kaggle/input/notebooks/kimthanh211005/notebook1700561abd/gr00t_x_embodiment_sim
- /kaggle/input/notebooks/kimthanh211005/notebookb6c37dbbd0/gr00t_x_embodiment_sim
- /kaggle/input/notebooks/kimthanh211005/notebookbcd29698a0/gr00t_x_embodiment_sim
- /kaggle/input/notebooks/kimthanh211005/notebookd3f6a10db4/gr00t_x_embodiment_sim_video_supplement

SUBSET_SOURCES:
- gr1_arms_only.CanSort
  data/meta: /kaggle/input/notebooks/kimthanh211005/notebook1700561abd/gr00t_x_embodiment_sim/gr1_arms_only.CanSort
  video paths: ['/kaggle/input/notebooks/kimthanh211005/notebook1700561abd/gr00t_x_embodiment_sim/gr1_arms_only.CanSort']
- gr1_arms_waist.CanToDrawer
  data/meta: /kaggle/input/notebooks/kimthanh211005/notebookb6c37dbbd0/gr00t_x_embodiment_sim/gr1_arms_waist.CanToDrawer
  video paths: ['/kaggle/input/notebooks/kimthanh211005/notebookd3f6a10db4/gr00t_x_embodiment_sim_video_supplement/gr1_arms_waist.CanToDrawer']
- gr1_arms_waist.CupToDrawer
  data/meta: /kaggle/input/notebooks/k

In [4]:
def folder_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 3)


scan_report = {
    "raw_roots": [str(p) for p in RAW_ROOTS],
    "output_root": str(OUTPUT_ROOT),
    "download_evidence": DOWNLOAD_EVIDENCE,
    "subsets": [],
}

for subset in SUBSETS:
    source_info = SUBSET_SOURCES[subset]
    subset_path = source_info["data_meta_path"]
    data_dir = subset_path / "data"
    meta_dir = subset_path / "meta"
    video_paths = source_info["video_paths"]
    video_files = []
    for video_path in video_paths:
        video_files.extend((video_path / "videos").rglob("*.mp4"))
    item = {
        "name": subset,
        "data_meta_path": str(subset_path),
        "video_paths": [str(p) for p in video_paths],
        "all_paths": [str(p) for p in source_info["all_paths"]],
        "exists": subset_path.exists(),
        "data_meta_size_gb": round(folder_size_gb(subset_path), 3),
        "video_size_gb": round(sum(folder_size_gb(p) for p in video_paths), 3),
        "has_data_dir": data_dir.exists(),
        "has_videos_dir": len(video_paths) > 0,
        "has_meta_dir": meta_dir.exists(),
        "num_parquet_files": len(list(data_dir.rglob("*.parquet"))) if data_dir.exists() else 0,
        "num_video_files": len(video_files),
    }
    scan_report["subsets"].append(item)

with open(OUTPUT_ROOT / "dataset_scan_report.json", "w", encoding="utf-8") as f:
    json.dump(scan_report, f, ensure_ascii=False, indent=2)

pd.DataFrame(scan_report["subsets"])


,name,data_meta_path,video_paths,all_paths,exists,data_meta_size_gb,video_size_gb,has_data_dir,has_videos_dir,has_meta_dir,num_parquet_files,num_video_files
0,gr1_arms_only.CanSort,/kaggle/input/notebooks/kimthanh211005/noteboo...,[/kaggle/input/notebooks/kimthanh211005/notebo...,[/kaggle/input/notebooks/kimthanh211005/notebo...,True,1.247,1.247,True,True,True,1000,1000
1,gr1_arms_waist.CanToDrawer,/kaggle/input/notebooks/kimthanh211005/noteboo...,[/kaggle/input/notebooks/kimthanh211005/notebo...,[/kaggle/input/notebooks/kimthanh211005/notebo...,True,1.494,6.198,True,True,True,10107,3000
2,gr1_arms_waist.CupToDrawer,/kaggle/input/notebooks/kimthanh211005/noteboo...,[/kaggle/input/notebooks/kimthanh211005/notebo...,[/kaggle/input/notebooks/kimthanh211005/notebo...,True,18.374,18.374,True,True,True,10036,10036


In [5]:
def read_subset_parquets(subset: str) -> pd.DataFrame:
    subset_path = SUBSET_SOURCES[subset]["data_meta_path"]
    data_dir = subset_path / "data"
    parquet_files = sorted(data_dir.rglob("*.parquet")) if data_dir.exists() else []
    if not parquet_files:
        raise FileNotFoundError(f"Không tìm thấy parquet files cho subset {subset}: {data_dir}")

    frames = []
    for parquet_path in tqdm(parquet_files, desc=f"Reading {subset}"):
        df = pd.read_parquet(parquet_path)
        df["__subset"] = subset
        df["__subset_source"] = str(subset_path)
        df["__parquet_file"] = str(parquet_path.relative_to(subset_path))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


all_frames = []
for subset in SUBSETS:
    all_frames.append(read_subset_parquets(subset))

if not all_frames:
    raise RuntimeError("Không đọc được subset nào. Hãy kiểm tra RAW_ROOTS/SUBSET_SOURCES và các Kaggle input đã Add.")

raw_df = pd.concat(all_frames, ignore_index=True)
print("Rows:", len(raw_df))
print("Columns:", len(raw_df.columns))
raw_df.head()


Reading gr1_arms_only.CanSort:   0%|          | 0/1000 [00:00<?, ?it/s]

Reading gr1_arms_waist.CanToDrawer:   0%|          | 0/10107 [00:00<?, ?it/s]

Reading gr1_arms_waist.CupToDrawer:   0%|          | 0/10036 [00:00<?, ?it/s]

Rows: 6192174
Columns: 14


,observation.state,action,timestamp,annotation.human.action.task_description,task_index,annotation.human.validity,episode_index,index,next.reward,next.done,__subset,__subset_source,__parquet_file,annotation.human.coarse_action
0,"[-0.006144224666329559, 0.10060920656063388, -...","[-0.0008818577638380776, 0.08654179422950317, ...",0.00,0.0,0,1.0,0,0,0.0,False,gr1_arms_only.CanSort,/kaggle/input/notebooks/kimthanh211005/noteboo...,data/chunk-000/episode_000000.parquet,NaN
1,"[-0.005014849323836754, 0.09771979213029992, -...","[0.006270111940881467, 0.08336182350310666, 0....",0.05,0.0,0,1.0,0,1,0.0,False,gr1_arms_only.CanSort,/kaggle/input/notebooks/kimthanh211005/noteboo...,data/chunk-000/episode_000000.parquet,NaN
2,"[-0.0025877249666917575, 0.09457997212407927, ...","[0.01329187207974061, 0.07946976304020072, 0.0...",0.10,0.0,0,1.0,0,2,0.0,False,gr1_arms_only.CanSort,/kaggle/input/notebooks/kimthanh211005/noteboo...,data/chunk-000/episode_000000.parquet,NaN
3,"[0.0008774730990542665, 0.09128270141399158, -...","[0.020210553521343263, 0.07521721549205151, 0....",0.15,0.0,0,1.0,0,3,0.0,False,gr1_arms_only.CanSort,/kaggle/input/notebooks/kimthanh211005/noteboo...,data/chunk-000/episode_000000.parquet,NaN
4,"[0.005122284556693562, 0.08777667061297888, -8...","[0.027059475796682962, 0.0708315971667002, 0.0...",0.20,0.0,0,1.0,0,4,0.0,False,gr1_arms_only.CanSort,/kaggle/input/notebooks/kimthanh211005/noteboo...,data/chunk-000/episode_000000.parquet,NaN


In [6]:
def choose_first_existing(columns: List[str], candidates: List[str]):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def is_numeric_series(series: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(series)


def numeric_vector_from_cell(value):
    if isinstance(value, np.ndarray):
        arr = value
    elif isinstance(value, (list, tuple)):
        arr = np.asarray(value)
    else:
        return None
    if arr.size == 0:
        return None
    if np.issubdtype(arr.dtype, np.number):
        return arr.astype(np.float32).reshape(-1)
    return None


def require_vector_column(df: pd.DataFrame, col: str, role: str) -> List[str]:
    if col not in df.columns:
        raise RuntimeError(
            f"Không tìm thấy cột {role} bắt buộc `{col}`. "
            f"Các cột hiện có: {', '.join(df.columns[:80])}"
        )
    sample = df[col].dropna().head(20)
    if sample.empty:
        raise RuntimeError(f"Cột {role} `{col}` tồn tại nhưng toàn bộ sample đầu là NaN/empty.")
    first_vec = None
    for value in sample:
        first_vec = numeric_vector_from_cell(value)
        if first_vec is not None:
            break
    if first_vec is None:
        raise RuntimeError(f"Cột {role} `{col}` không phải vector numeric, chưa dùng được cho train.")
    return [col]


columns = list(raw_df.columns)
episode_col = choose_first_existing(columns, ["episode_index", "episode_id", "episode", "index.episode"])
frame_col = choose_first_existing(columns, ["frame_index", "frame_id", "timestamp", "index.frame"])

if episode_col is None:
    raise RuntimeError(
        "Không detect được episode column. Các cột hiện có: " + ", ".join(columns[:80])
    )

if frame_col is None:
    raw_df["__frame_index"] = raw_df.groupby(["__subset", episode_col]).cumcount()
    frame_col = "__frame_index"

# Dùng đúng schema GR00T/LeRobot: action target chỉ là cột `action`.
# Không đưa annotation/task id vào action vector, vì notebook 03 phải học motor action thuần.
action_cols = require_vector_column(raw_df, "action", "action")
state_cols = require_vector_column(raw_df, "observation.state", "state")
ignored_action_like_cols = [
    col for col in columns
    if "action" in col.lower() and col not in action_cols
]

print("episode_col:", episode_col)
print("frame_col:", frame_col)
print("action_cols:", action_cols)
print("state_cols:", state_cols)
print("ignored_action_like_cols:", ignored_action_like_cols)


episode_col: episode_index
frame_col: timestamp
action_cols: ['action']
state_cols: ['observation.state']
ignored_action_like_cols: ['annotation.human.action.task_description', 'annotation.human.coarse_action']


In [7]:
def flatten_numeric_values(row: pd.Series, cols: List[str]) -> np.ndarray:
    parts = []
    for col in cols:
        value = row[col]
        vec = numeric_vector_from_cell(value)
        if vec is not None:
            parts.append(vec)
        elif pd.notna(value) and np.isscalar(value):
            try:
                parts.append(np.asarray([float(value)], dtype=np.float32))
            except (TypeError, ValueError):
                pass
    if not parts:
        return np.asarray([], dtype=np.float32)
    return np.concatenate(parts).astype(np.float32)


def build_matrix(df: pd.DataFrame, cols: List[str], name: str) -> np.ndarray:
    vectors = []
    expected_dim = None
    bad_dims = []
    empty_rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Building {name}"):
        vec = flatten_numeric_values(row, cols)
        if vec.size == 0:
            empty_rows.append(idx)
            if len(empty_rows) >= 5:
                break
            continue
        if expected_dim is None:
            expected_dim = vec.size
        if vec.size != expected_dim:
            bad_dims.append({
                "row_index": int(idx),
                "subset": str(row.get("__subset", "")),
                "episode": str(row.get(episode_col, "")),
                "actual_dim": int(vec.size),
                "expected_dim": int(expected_dim),
            })
            if len(bad_dims) >= 5:
                break
            continue
        vectors.append(vec)

    if empty_rows:
        raise RuntimeError(f"{name} có row rỗng tại indices mẫu: {empty_rows}")
    if bad_dims:
        raise RuntimeError(f"{name} dimension không đồng nhất. Ví dụ lỗi: {bad_dims}")
    if not vectors:
        raise RuntimeError(f"Không tạo được matrix cho {name} từ columns={cols}")
    return np.stack(vectors).astype(np.float32)


# Sort before optional frame cap and split.
work_df = raw_df.sort_values(["__subset", episode_col, frame_col]).reset_index(drop=True)

if MAX_FRAMES_PER_EPISODE is not None:
    work_df = (
        work_df.groupby(["__subset", episode_col], group_keys=False)
        .head(MAX_FRAMES_PER_EPISODE)
        .reset_index(drop=True)
    )

actions_all = build_matrix(work_df, action_cols, "actions")

states_all = build_matrix(work_df, state_cols, "states")

subset_row_counts = work_df["__subset"].value_counts().sort_index().to_dict()
subset_episode_counts = work_df.groupby("__subset")[episode_col].nunique().sort_index().to_dict()

print("Prepared rows:", len(work_df))
print("Action shape:", actions_all.shape)
print("State shape:", states_all.shape)
print("Rows by subset:", subset_row_counts)
print("Episodes by subset:", subset_episode_counts)


Building actions:   0%|          | 0/6192174 [00:00<?, ?it/s]

Building states:   0%|          | 0/6192174 [00:00<?, ?it/s]

Prepared rows: 6192174
Action shape: (6192174, 44)
State shape: (6192174, 44)
Rows by subset: {'gr1_arms_only.CanSort': 316878, 'gr1_arms_waist.CanToDrawer': 3210426, 'gr1_arms_waist.CupToDrawer': 2664870}
Episodes by subset: {'gr1_arms_only.CanSort': 1000, 'gr1_arms_waist.CanToDrawer': 10107, 'gr1_arms_waist.CupToDrawer': 10036}


In [8]:
episode_keys = sorted(set(zip(work_df["__subset"].astype(str), work_df[episode_col].astype(str))))
rng = random.Random(SEED)
rng.shuffle(episode_keys)

num_train = max(1, int(len(episode_keys) * TRAIN_RATIO))
if len(episode_keys) > 1:
    num_train = min(num_train, len(episode_keys) - 1)

train_episode_keys = set(episode_keys[:num_train])
test_episode_keys = set(episode_keys[num_train:])

is_train = np.array([
    (str(row["__subset"]), str(row[episode_col])) in train_episode_keys
    for _, row in work_df.iterrows()
])

train_df = work_df.loc[is_train].reset_index(drop=True)
test_df = work_df.loc[~is_train].reset_index(drop=True)
actions_train = actions_all[is_train]
actions_test = actions_all[~is_train]

states_train = states_all[is_train] if states_all is not None else None
states_test = states_all[~is_train] if states_all is not None else None

with open(OUTPUT_ROOT / "splits_train_episodes.json", "w", encoding="utf-8") as f:
    json.dump([{"subset": s, "episode": e} for s, e in sorted(train_episode_keys)], f, ensure_ascii=False, indent=2)

with open(OUTPUT_ROOT / "splits_test_episodes.json", "w", encoding="utf-8") as f:
    json.dump([{"subset": s, "episode": e} for s, e in sorted(test_episode_keys)], f, ensure_ascii=False, indent=2)

overlap = train_episode_keys.intersection(test_episode_keys)
assert not overlap, f"Episode leakage detected: {overlap}"

print("Episodes:", len(episode_keys))
print("Train episodes:", len(train_episode_keys), "Test episodes:", len(test_episode_keys))
print("Train rows:", len(train_df), "Test rows:", len(test_df))


Episodes: 21143
Train episodes: 16914 Test episodes: 4229
Train rows: 4952021 Test rows: 1240153


In [9]:
def make_json_safe_df(df: pd.DataFrame) -> pd.DataFrame:
    # Keep scalar/object metadata that parquet can store reliably. Large nested arrays are saved separately.
    keep_cols = []
    for col in df.columns:
        sample = df[col].dropna().head(5)
        if sample.empty:
            keep_cols.append(col)
            continue
        has_nested = any(isinstance(v, (list, tuple, np.ndarray, dict)) for v in sample)
        if not has_nested:
            keep_cols.append(col)
    return df[keep_cols].copy()


train_samples = make_json_safe_df(train_df)
test_samples = make_json_safe_df(test_df)
sample_index = make_json_safe_df(work_df[["__subset", episode_col, frame_col, "__parquet_file"]].copy())

train_samples.to_parquet(OUTPUT_ROOT / "train_samples.parquet", index=False)
test_samples.to_parquet(OUTPUT_ROOT / "test_samples.parquet", index=False)
sample_index.to_parquet(OUTPUT_ROOT / "sample_index.parquet", index=False)

np.save(OUTPUT_ROOT / "actions_train.npy", actions_train)
np.save(OUTPUT_ROOT / "actions_test.npy", actions_test)

if states_train is not None and states_test is not None:
    np.save(OUTPUT_ROOT / "states_train.npy", states_train)
    np.save(OUTPUT_ROOT / "states_test.npy", states_test)

print("Saved prepared artifacts to", OUTPUT_ROOT)
print(sorted(p.name for p in OUTPUT_ROOT.iterdir()))


Saved prepared artifacts to /kaggle/working/gr00t_prepared_3subsets
['actions_test.npy', 'actions_train.npy', 'dataset_scan_report.json', 'sample_index.parquet', 'splits_test_episodes.json', 'splits_train_episodes.json', 'states_test.npy', 'states_train.npy', 'test_samples.parquet', 'train_samples.parquet']


In [10]:
def dir_size_mb(path: Path) -> float:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 2)


prepare_report = {
    "raw_roots": [str(p) for p in RAW_ROOTS],
    "output_root": str(OUTPUT_ROOT),
    "download_evidence": DOWNLOAD_EVIDENCE,
    "subset_sources": {
        subset: {
            "data_meta_path": str(info["data_meta_path"]),
            "video_paths": [str(p) for p in info["video_paths"]],
            "all_paths": [str(p) for p in info["all_paths"]],
        }
        for subset, info in SUBSET_SOURCES.items()
    },
    "subsets_requested": SUBSETS,
    "subsets_found": sorted(work_df["__subset"].unique().tolist()),
    "seed": SEED,
    "train_ratio": TRAIN_RATIO,
    "max_frames_per_episode": MAX_FRAMES_PER_EPISODE,
    "episode_col": episode_col,
    "frame_col": frame_col,
    "action_cols": action_cols,
    "ignored_action_like_cols": ignored_action_like_cols,
    "state_cols": state_cols,
    "subset_row_counts": {k: int(v) for k, v in subset_row_counts.items()},
    "subset_episode_counts": {k: int(v) for k, v in subset_episode_counts.items()},
    "num_episodes": len(episode_keys),
    "num_train_episodes": len(train_episode_keys),
    "num_test_episodes": len(test_episode_keys),
    "num_train_samples": int(len(train_df)),
    "num_test_samples": int(len(test_df)),
    "action_dim": int(actions_all.shape[1]),
    "state_dim": int(states_all.shape[1]),
    "output_size_mb": round(dir_size_mb(OUTPUT_ROOT), 2),
}

with open(OUTPUT_ROOT / "prepare_report.json", "w", encoding="utf-8") as f:
    json.dump(prepare_report, f, ensure_ascii=False, indent=2)

print(json.dumps(prepare_report, ensure_ascii=False, indent=2))


{
  "raw_roots": [
    "/kaggle/input/notebooks/kimthanh211005/notebook1700561abd/gr00t_x_embodiment_sim",
    "/kaggle/input/notebooks/kimthanh211005/notebookb6c37dbbd0/gr00t_x_embodiment_sim",
    "/kaggle/input/notebooks/kimthanh211005/notebookbcd29698a0/gr00t_x_embodiment_sim",
    "/kaggle/input/notebooks/kimthanh211005/notebookd3f6a10db4/gr00t_x_embodiment_sim_video_supplement"
  ],
  "output_root": "/kaggle/working/gr00t_prepared_3subsets",
  "download_evidence": [
    {
      "notebook_id": "notebook1700561abd",
      "subset": "gr1_arms_only.CanSort",
      "content": "data + meta + videos",
      "duration": "6 phút",
      "report": "download_report_1_5.json"
    },
    {
      "notebook_id": "notebookbcd29698a0",
      "subset": "gr1_arms_waist.CupToDrawer",
      "content": "data + meta + videos",
      "duration": "1 tiếng 40 phút",
      "report": "download_report_1_5.json"
    },
    {
      "notebook_id": "notebookb6c37dbbd0",
      "subset": "gr1_arms_waist.CanToDrawe

## Sau Khi Chạy Xong

1. Trước khi chạy, Add Input đủ 4 output datasets đã tải:
   - `notebook1700561abd`: `gr1_arms_only.CanSort`, 6 phút.
   - `notebookbcd29698a0`: `gr1_arms_waist.CupToDrawer`, 1 tiếng 40 phút.
   - `notebookb6c37dbbd0`: `gr1_arms_waist.CanToDrawer` data + meta, 24 phút.
   - `notebookd3f6a10db4`: `gr1_arms_waist.CanToDrawer` video chunks 0,1,2, 20 phút.
2. Mở tab **Output** của Kaggle notebook.
3. Kiểm tra có thư mục `gr00t_prepared_3subsets`.
4. Bấm **Save Version -> Save & Run All**.
5. Dùng output này làm Kaggle Dataset input cho notebook tiếp theo: `03_train_dit_from_scratch.ipynb`.

Các file quan trọng cần có:

```text
prepare_report.json
splits_train_episodes.json
splits_test_episodes.json
train_samples.parquet
test_samples.parquet
actions_train.npy
actions_test.npy
```
